In [13]:
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier


In [14]:
MODEL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\models"

mobilevit_features = np.load(f"{MODEL_DIR}/mobilevit_features.npy")
swinB_features = np.load(f"{MODEL_DIR}/swinB_features.npy")   # 👈 Swin-B
labels = np.load(f"{MODEL_DIR}/labels.npy")

print("MobileViT:", mobilevit_features.shape)
print("Swin-B:", swinB_features.shape)
print("Labels:", labels.shape)


MobileViT: (6939, 640)
Swin-B: (6939, 1024)
Labels: (6939,)


In [15]:
X = np.concatenate([mobilevit_features, swinB_features], axis=1)
y = labels

print("Final feature shape:", X.shape)


Final feature shape: (6939, 1664)


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])


Train samples: 5551
Test samples: 1388


In [5]:
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.04,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    gamma=0.15,
    reg_alpha=0.15,
    reg_lambda=1.2,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("XGBoost (Swin-B) training completed ✅")


XGBoost (Swin-B) training completed ✅


In [17]:
y_pred = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


NotFittedError: need to call fit or load_model beforehand

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [19]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

scale_pos_weight = class_weights[1] / class_weights[0]
print("scale_pos_weight:", scale_pos_weight)


scale_pos_weight: 1.2150837988826817


In [20]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)
X_train = selector.fit_transform(X_train)
X_test = selector.transform(X_test)

print("Reduced feature shape:", X_train.shape)


Reduced feature shape: (5551, 1664)


In [25]:
xgb_model = XGBClassifier(
    n_estimators=600,
    max_depth=7,
    learning_rate=0.035,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=4,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
print("Model trained on shape:", X_train.shape)



Model trained on shape: (5551, 1664)


In [26]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

y_pred = xgb_model.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(y_test, y_pred)
print("Final Accuracy:", acc)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


X_train shape: (5551, 1664)
X_test shape: (1388, 1664)
y_train shape: (5551,)
y_test shape: (1388,)
Final Accuracy: 0.9329971181556196

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.94      0.94       761
           1       0.93      0.92      0.93       627

    accuracy                           0.93      1388
   macro avg       0.93      0.93      0.93      1388
weighted avg       0.93      0.93      0.93      1388



In [1]:
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

MODEL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\models"
RESULTS_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\notebooks\results"

# Load features
mobilevit = np.load(f"{MODEL_DIR}/mobilevit_features.npy")
swinB = np.load(f"{MODEL_DIR}/swinB_features.npy")
labels = np.load(f"{MODEL_DIR}/labels.npy")

X = np.concatenate([mobilevit, swinB], axis=1)
y = labels

# Split ONCE (important!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Save split (VERY IMPORTANT)
np.save(f"{RESULTS_DIR}/X_test_swinB.npy", X_test)
np.save(f"{RESULTS_DIR}/y_test_swinB.npy", y_test)

# Class balance
class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
scale_pos_weight = class_weights[1] / class_weights[0]

# XGBoost
xgb_swinB = XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_swinB.fit(X_train, y_train)

# Predictions
y_pred = xgb_swinB.predict(X_test)
y_prob = xgb_swinB.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Save everything
joblib.dump(xgb_swinB, f"{MODEL_DIR}/xgb_mobilevit_swinB.pkl")

np.save(f"{RESULTS_DIR}/swinB_y_test.npy", y_test)
np.save(f"{RESULTS_DIR}/swinB_y_pred.npy", y_pred)
np.save(f"{RESULTS_DIR}/swinB_y_prob.npy", y_prob)

print("Swin-B XGBoost saved ✅")


Accuracy: 0.9344380403458213
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       761
           1       0.93      0.92      0.93       627

    accuracy                           0.93      1388
   macro avg       0.93      0.93      0.93      1388
weighted avg       0.93      0.93      0.93      1388

Swin-B XGBoost saved ✅
